# Fitting an Exponential Distribution to POT Data — a Python Port of Tony Ladson's R Method

**Companion notebook to:** _Fitting a Probability Model to POT Data — a Python Port of Tony Ladson's R Method_

**Source:** [Fitting a probability model to POT data](https://tonyladson.wordpress.com/2019/03/25/fitting-a-probability-model-to-pot-data/) — Tony Ladson, 25 March 2019. R source: [gist.github.com/TonyLadson/5b01838fef1140293397e23eebe12079](https://gist.github.com/TonyLadson/5b01838fef1140293397e23eebe12079) (`Pot_fit.R`).

This is a line-for-line Python port of Ladson's R script, run on the exact same data (the Styx River at Jeogla partial-duration series, 47 peaks) and checked against his own published numbers at every step. Where a number is checked against his output, it's marked as such below — this notebook only claims to reproduce what was actually checked, not what merely looks plausible.

Method: fit an exponential distribution to a Partial Duration Series via L-moments (Wang 1996), following ARR 2016/2019 Book 3 Section 2.8.11, and convert between AEP (Annual Exceedance Probability) and EY (Exceedances per Year) correctly for a POT series — a plotting-position formula alone isn't valid here, since more than one exceedance can occur per year.

In [1]:
import numpy as np
from scipy import stats
from math import comb
import matplotlib.pyplot as plt

print('numpy:', np.__version__)
print('scipy:', __import__('scipy').__version__)

numpy: 2.4.6
scipy: 1.17.1


## 1. The data

The Styx River at Jeogla partial series, as published in Ladson's gist — 47 peaks over threshold, in m³/s.

In [2]:
styx = np.array([74.0, 79.9, 85.2, 88.6, 91.7, 92.2, 92.2, 98.0, 105.0, 108.0, 111.0,
                  117.0, 117.0, 118.0, 119.0, 126.0, 129.0, 129.0, 134.0, 149.0, 150.0, 164.0,
                  186.0, 190.0, 194.0, 196.0, 206.0, 220.0, 221.0, 235.0, 238.0, 255.0, 255.0,
                  258.0, 283.0, 294.0, 300.0, 301.0, 309.0, 315.0, 405.0, 411.0, 436.0, 513.0, 521.0, 541.0, 878.0])

print(f'n = {len(styx)}')
assert len(styx) == 47, "Ladson's post: 'length(styx) #47'"
print('Matches Ladson\'s R output: length(styx) #47')

n = 47
Matches Ladson's R output: length(styx) #47


## 2. L-moments and the exponential parameters

Ladson's `L2()` function is the direct sample L-moment estimator from Wang (1996). The exponential distribution is then parameterised by scale `beta = 2*L2` and location `q_star = L1 - beta` (L1 is just the mean).

In [3]:
def l2_wang(q):
    """Second L-moment, direct sample estimator (Wang, Q.J. 1996,
    'Direct sample estimates of L moments', Water Resources Research 32(12))."""
    q = np.sort(np.asarray(q, dtype=float))
    n = len(q)
    i = np.arange(n)
    weights = i - (n - 1 - i)
    return 0.5 * (1 / comb(n, 2)) * np.sum(weights * q)

L1 = styx.mean()
L2 = l2_wang(styx)
beta = 2 * L2
q_star = L1 - beta

print(f'L1     = {L1:.2f}      (Ladson: 226.36)')
print(f'L2     = {L2:.2f}       (Ladson: 79.12)')
print(f'beta   = {beta:.5f}  (Ladson: 158.240)')
print(f'q_star = {q_star:.5f}  (Ladson: 68.11748)')

L1     = 226.36      (Ladson: 226.36)
L2     = 79.12       (Ladson: 79.12)
beta   = 158.23996  (Ladson: 158.240)
q_star = 68.11748  (Ladson: 68.11748)


## 3. AEP <-> EY conversion, and the flood quantile function

This is the part of Ladson's post worth reading in full: a Cunnane (or any) plotting position formula gives an empirical AEP for an *annual maximum* series, where by construction there's exactly one value per year. A POT series can have several peaks in one year and none in another, so the right quantity is EY (expected number of exceedances per year) — related to AEP through the Poisson assumption that exceedances occur independently at a constant rate:

- `AEP = 1 - exp(-EY)`
- `EY = -log(1 - AEP)`

The fitted exponential's quantile function is then evaluated directly in EY, not AEP.

In [4]:
def ey_to_aep(EY):
    return (np.exp(EY) - 1) / np.exp(EY)  # == 1 - exp(-EY)

def aep_to_ey(AEP):
    return -np.log(1 - AEP)

def flood_quantile(EY, beta, q_star, nu=1.0):
    return q_star - beta * np.log(EY / nu)

standard_EY = np.array([1, 0.69, 0.5, 0.22, 0.2, 0.11, 0.05, 0.02, 0.01])
AEP = ey_to_aep(standard_EY)
ARI = 1 / standard_EY
Q = flood_quantile(standard_EY, beta, q_star)

ladson_Q = [68.1, 127.0, 178.0, 308.0, 323.0, 417.0, 542.0, 687.0, 797.0]
print(f'{"EY":>6} {"AEP":>8} {"ARI":>8} {"Q":>8}   {"Ladson Q":>10}')
for ey, aep, ari, q, lq in zip(standard_EY, AEP, ARI, Q, ladson_Q):
    assert abs(q - lq) < 0.5, f'mismatch at EY={ey}: got {q}, Ladson has {lq}'
    print(f'{ey:6.2f} {aep:8.4f} {ari:8.2f} {q:8.1f}   {lq:10.1f}')

print()
print('All 9 quantiles match Ladson\'s published table to the precision he reported.')
print(f'Explicit check value from his post: EY=0.01 -> {flood_quantile(0.01, beta, q_star):.4f} (Ladson: 796.8394)')

    EY      AEP      ARI        Q     Ladson Q
  1.00   0.6321     1.00     68.1         68.1
  0.69   0.4984     1.45    126.8        127.0
  0.50   0.3935     2.00    177.8        178.0
  0.22   0.1975     4.55    307.7        308.0
  0.20   0.1813     5.00    322.8        323.0
  0.11   0.1042     9.09    417.4        417.0
  0.05   0.0488    20.00    542.2        542.0
  0.02   0.0198    50.00    687.2        687.0
  0.01   0.0100   100.00    796.8        797.0

All 9 quantiles match Ladson's published table to the precision he reported.
Explicit check value from his post: EY=0.01 -> 796.8394 (Ladson: 796.8394)


## 4. Bootstrap confidence intervals

Ladson uses R's `boot::boot` with 5,000 resamples and BCa (bias-corrected and accelerated) confidence intervals. `scipy.stats.bootstrap` supports the same BCa method directly, so this doesn't need a hand-rolled implementation.

Point estimates match exactly (they're deterministic). The confidence interval *bounds* won't match his bit-for-bit — different RNG draws — but should land in the same range for the same method on the same data.

In [5]:
def exp_params(q):
    q = np.asarray(q)
    b = 2 * l2_wang(q)
    qs = q.mean() - b
    return b, qs

def beta_stat(sample, axis=-1):
    return np.apply_along_axis(lambda s: exp_params(s)[0], axis, sample)

def qstar_stat(sample, axis=-1):
    return np.apply_along_axis(lambda s: exp_params(s)[1], axis, sample)

def q01_stat(sample, axis=-1):
    return np.apply_along_axis(lambda s: flood_quantile(0.01, *exp_params(s)), axis, sample)

rng = np.random.default_rng(20260901)
print(f'{"statistic":14s} {"point":>10s}  {"95% CI":>22s}   {"Ladson":s}')
for name, fn, ladson in [
    ('beta', beta_stat, 'est=158.24'),
    ('q_star', qstar_stat, 'est=68.12'),
    ('Q at EY=0.01', q01_stat, 'est=796.84, CI=(628, 1132)'),
]:
    res = stats.bootstrap((styx,), fn, n_resamples=5000, method='BCa', confidence_level=0.95, random_state=rng)
    point = fn(styx.reshape(1, -1), axis=1)[0]
    print(f'{name:14s} {point:10.2f}  ({res.confidence_interval.low:8.2f}, {res.confidence_interval.high:8.2f})   {ladson}')

statistic           point                  95% CI   Ladson
beta               158.24  (  120.54,   234.11)   est=158.24
q_star              68.12  (   37.57,    89.79)   est=68.12
Q at EY=0.01       796.84  (  626.03,  1108.99)   est=796.84, CI=(628, 1132)


The bootstrap CI on the 1% quantile — (626, 1109) here vs. Ladson's (628, 1132) — is close but not identical, exactly as expected for two independent resampling runs of the same method.

## 5. The flood frequency plot

Reproducing Ladson's plot: peaks on the Cunnane plotting position (as EY, since this is a POT series), the fitted exponential curve, and a bootstrap confidence band.

In [6]:
def cunnane_ey(q_ascending):
    """Cunnane plotting position, interpreted as EY for a POT series (not AEP).
    Rank 1 = smallest value -> smallest EY; rank n = largest value -> largest EY."""
    n = len(q_ascending)
    ranks = np.arange(1, n + 1)
    return (ranks - 0.4) / (n + 0.2)

# Ladson pairs Q sorted DESCENDING (largest peak first) against EY computed
# from ascending-sorted ranks -- so the largest peak lines up with the
# smallest EY (rarest event), which is the correct physical pairing.
q_sorted_desc = np.sort(styx)[::-1]
ey_for_plot = cunnane_ey(np.sort(styx))

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(ey_for_plot, q_sorted_desc, s=18, color='dimgray', zorder=3, label='Styx River POT peaks')

EY_seq = np.logspace(np.log10(0.01), np.log10(1), 200)
fit_curve = flood_quantile(EY_seq, beta, q_star)
ax.plot(EY_seq, fit_curve, color='steelblue', lw=2, label='Fitted exponential (L-moments)')

# Bootstrap band at a coarser set of EY points (cheaper than Ladson's per-point
# bootstrap, but the same idea)
band_EY = np.logspace(np.log10(0.01), np.log10(1), 12)
lo, hi = [], []
for ey in band_EY:
    def q_at_ey(sample, axis=-1, ey=ey):
        return np.apply_along_axis(lambda s: flood_quantile(ey, *exp_params(s)), axis, sample)
    r = stats.bootstrap((styx,), q_at_ey, n_resamples=1000, method='BCa', confidence_level=0.95, random_state=rng)
    lo.append(r.confidence_interval.low)
    hi.append(r.confidence_interval.high)
ax.plot(band_EY, lo, color='steelblue', lw=1, linestyle='--', alpha=0.7, label='95% bootstrap CI')
ax.plot(band_EY, hi, color='steelblue', lw=1, linestyle='--', alpha=0.7)

ax.set_xscale('log')
ax.set_xlabel('EY (Exceedances per year, log scale)')
ax.set_ylabel('Peak flow (m$^3$/s)')
ax.set_title('Styx River at Jeogla -- exponential fit to POT data', fontsize=12)
ax.grid(True, which='both', alpha=0.3)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../../images/2026-09_pot-exponential-fit-styx.png', dpi=150, bbox_inches='tight')
plt.show()

<Figure size ... with Axes>

## 6. What this translation checked, and what it didn't

**Checked against Ladson's own published R output:** `n=47`, `L1`, `L2`, `beta`, `q_star`, all 9 standard-EY flood quantiles (including his explicit `EY=0.01 -> 796.8394` check value), and the bootstrap CI landing in the same range as his.

**Not independently checked:** whether the exponential distribution is actually the right choice for the Styx River series specifically (that's a modelling decision Ladson's post inherits from the ARR worked example, not something this notebook re-derives), and the BCa bootstrap implementation's small-sample behaviour beyond agreeing with one reference case.

## References

- Ladson, A.R. (2019). [Fitting a probability model to POT data](https://tonyladson.wordpress.com/2019/03/25/fitting-a-probability-model-to-pot-data/). R source: [gist.github.com/TonyLadson/5b01838fef1140293397e23eebe12079](https://gist.github.com/TonyLadson/5b01838fef1140293397e23eebe12079).
- Wang, Q.J. (1996). Direct sample estimates of L moments. *Water Resources Research* 32(12): 3617-3619.
- Ball, J. et al. (2019). *Australian Rainfall and Runoff.* Book 3, Section 2.8.11.